In [10]:
# Блок 1: Импорты
import time
import numpy as np
from gym_art.quadrotor_multi.quadrotor_multi import QuadrotorEnvMulti
from gym_art.quadrotor_multi.quad_experience_replay import ExperienceReplayWrapper
import gym

from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn



Полный список коэффициентов наград:

- pos: 1.0 - за позиционирование
- effort: 0.05 - за затраченные усилия
- action_change: 0.0 - за изменение действий
- crash: 1.0 - за столкновения
- orient: 1.0 - за ориентацию
- yaw: 0.0 - за рыскание
- rot: 0.0 - за вращение
- attitude: 0.0 - за положение
- spin: 0.1 - за вращение вокруг оси
- vel: 0.0 - за скорость
- quadcol_bin: 5.0 - за столкновения между квадрокоптерами
- quadcol_bin_smooth_max: 4.0 - сглаженный максимум столкновений
- quadcol_bin_obst: 5.0 - за столкновения с препятствиями

In [11]:
def create_env(num_agents, 
               use_numba=False, 
               use_replay_buffer=False, 
               episode_duration=15, 
               quads_render=False
               ):
    quad = 'Crazyflie'
    dyn_randomize_every = dyn_randomization_ratio = None

    episode_duration = episode_duration  # seconds

    raw_control = raw_control_zero_middle = True

    sampler_1 = None
    if dyn_randomization_ratio is not None:
        sampler_1 = dict(type="RelativeSampler", noise_ratio=dyn_randomization_ratio, sampler="normal")

    sense_noise = 'default'

    dynamics_change = dict(noise=dict(thrust_noise_ratio=0.05), damp=dict(vel=0, omega_quadratic=0))
   
    # Используем коэффициенты наград по умолчанию
    rew_coeff = dict(
        pos=1.,                     # награда за точность позиционирования
        effort=0.05,                # штраф за использование энергии
        action_change=0., # штраф за резкие изменения управления
        crash=1.,         # штраф за столкновения
        orient=1.,        # награда за правильную ориентацию
        yaw=0.,          # награда за контроль рыскания
        rot=0.,          # награда за вращение
        attitude=0.,      # награда за положение
        spin=0.1,        # штраф за вращение вокруг оси
        vel=0.,          # награда за контроль скорости
        quadcol_bin=5.,  # штраф за столкновения между квадрокоптерами
        quadcol_bin_smooth_max=4., # сглаженный штраф за столкновения
        quadcol_bin_obst=5.       # штраф за столкновения с препятствиями
        )
   # Создание среды с полным набором параметров
    env = QuadrotorEnvMulti(
        # Основные параметры
        num_agents=num_agents,
        dynamics_params=quad,
        raw_control=raw_control,
        raw_control_zero_middle=raw_control_zero_middle,
        
        # Параметры динамики и шума
        dynamics_randomize_every=dyn_randomize_every,
        dynamics_change=dynamics_change,
        dyn_sampler_1=sampler_1,
        sense_noise=sense_noise,
        init_random_state=False,
        
        # Временные параметры
        ep_time=episode_duration,
        
        #коэффициенты наград
        rew_coeff=rew_coeff,
        # Параметры вычислений
        use_numba=use_numba,
        use_replay_buffer=use_replay_buffer,
        
        # Параметры наблюдений
        obs_repr='xyz_vxyz_R_omega',  # Формат наблюдений: позиция, скорость, ориентация, угл.скорость
                
        # Параметры взаимодействия агентов
        neighbor_visible_num=0,  # Количество видимых соседей
        neighbor_obs_type='none',  # Тип наблюдений за соседями
        
        # Параметры столкновений
        collision_hitbox_radius=0.5,  # Радиус столкновений
        collision_falloff_radius=1.0,  # Радиус затухания столкновений
        
        # Параметры препятствий
        use_obstacles=False,  # Использование препятствий
        obst_density=0.0,    # Плотность препятствий
        obst_size=0.5,       # Размер препятствий
        obst_spawn_area=[8.0, 8.0],  # Область появления препятствий
        
        # Дополнительные физические эффекты
        use_downwash=False,  # Эффект воздушного потока
        
        # Параметры среды
        quads_mode='static_same_goal',  # Режим работы квадрокоптеров
        room_dims=[8.0, 8.0, 5.0],      # Размеры комнаты [x, y, z]
        
        # Параметры визуализации
        quads_view_mode=['topdown','side'],  # Режим отображения
        # [topdown, global, chase, side, corner0, corner1, corner2, corner3, topdownfollow]
        quads_render=quads_render            # Включение рендеринга
    )
    return env

In [12]:
    # 1. Создаем среду с базовыми параметрами
num_agents = 1  # Начнем с 2 квадрокоптеров
env = create_env(
        num_agents=num_agents,
        use_numba=False,
        episode_duration=7
    )



Observation components: ['xyz', 'vxyz', 'R', 'omega']


In [13]:
    # 2. Выводим информацию о среде
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Number of agents: {env.num_agents}")
    


Observation space: Box([ -8.  -8.  -5.  -3.  -3.  -3.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.
  -1. -40. -40. -40.], [ 8.  8.  5.  3.  3.  3.  1.  1.  1.  1.  1.  1.  1.  1.  1. 40. 40. 40.], (18,), float32)
Action space: Box(-1.0, 1.0, (4,), float32)
Number of agents: 1


Формат Box(min_values, max_values, shape, dtype) показывает:

1. Первые 3 значения [-8, -8, -5] до [8, 8, 5]:
- Диапазон возможных позиций x, y, z
- Соответствует размерам комнаты room_dims=[8.0, 8.0, 5.0]
2. Следующие 3 значения [-3, -3, -3] до [3, 3, 3]:
- Диапазон возможных скоростей vx, vy, vz
- Ограничение максимальной скорости ±3 м/с
3. Следующие 9 значений [-1,...,-1] до [1,...,1]:
- Элементы матрицы поворота R (3x3)
- Нормализованы в диапазоне [-1, 1]
4. Последние 3 значения [-40, -40, -40] до [40, 40, 40]:
- Угловые скорости ωx, ωy, ωz
- Ограничение максимальной угловой скорости ±40 рад/с

Shape (18,) означает, что вектор наблюдений содержит 18 значений. float32 - тип данных с плавающей точкой.

Box(-1.0, 1.0, (4,), float32) означает:

Размерность (4,) - управляем 4 моторами квадрокоптера
Диапазон [-1.0, 1.0] - нормализованная тяга для каждого мотора:
-1.0: минимальная тяга
0.0: средняя тяга
1.0: максимальная тяга
float32 - тип данных с плавающей точкой
Управление происходит через изменение тяги каждого из 4 моторов:

Мотор 1: передний правый
Мотор 2: задний левый
Мотор 3: передний левый
Мотор 4: задний правый
Комбинируя разные значения тяги моторов, можно выполнять все базовые маневры:

Взлет/посадка
Наклоны вперед/назад/вбок
Вращение вокруг вертикальной оси
Сложные комбинированные движения

In [5]:
# 3. Запускаем тестовый эпизод
obs = env.reset()

print("Начальное состояние после сброса:")
print(f"Observation shape: {len(obs[0])}")
print(f"Position: {obs[0][:3]}")  # xyz координаты
print(f"Velocity: {obs[0][3:6]}")  # скорости по осям
print(f"Rotation matrix: {obs[0][6:15]}")  # матрица поворота
print(f"Angular velocity: {obs[0][15:18]}")  # угловые скорости

# Также можно посмотреть границы пространства
print("\nГраницы пространства:")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

total_reward = 0

for step in range(30):
    # Генерируем случайные действия для каждого агента
    actions = [env.action_space.sample() for _ in range(num_agents)]
    
    # Выполняем шаг в среде
    obs, rewards, dones, infos = env.step(actions)
    
    # Выводим информацию каждые 20 шагов
    if step % 10 == 0:
        print(f"\nStep {step}:")
        print(f"Rewards: {rewards}")
        print(f"Actions: {actions}")  # Добавлен вывод действий
        print(f"Positions: {[o[:3] for o in obs]}")  # Первые 3 значения - позиция
        print(f"Velocities: {[o[3:6] for o in obs]}")  # Следующие 3 - скорость
        print(type(infos))
        print(f"Infos: {infos}")
        
    if any(dones):
        print("\nEpisode finished")
        break

env.close()


Начальное состояние после сброса:
Observation shape: 18
Position: [-0.48863532  1.38710953  0.829483  ]
Velocity: [ 0.00052702 -0.00627421 -0.01724411]
Rotation matrix: [ 0.61152337  0.79122637  0.         -0.79122637  0.61152337 -0.
 -0.          0.          1.        ]
Angular velocity: [ 1.31691395e-04  5.16884361e-05 -2.64802689e-04]

Границы пространства:
Observation space: Box([ -8.  -8.  -5.  -3.  -3.  -3.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.
  -1. -40. -40. -40.], [ 8.  8.  5.  3.  3.  3.  1.  1.  1.  1.  1.  1.  1.  1.  1. 40. 40. 40.], (18,), float32)
Action space: Box(-1.0, 1.0, (4,), float32)

Step 0:
Rewards: [-0.0037299554709862193]
Actions: [array([-0.80802673, -0.59523016,  0.06469189, -0.3728606 ], dtype=float32)]
Positions: [array([-0.48649259,  1.38510071,  0.8313435 ])]
Velocities: [array([ 0.00792443, -0.02069398, -0.0951094 ])]
<class 'list'>
Infos: [{'rewards': {'rew_main': -0.008403355545391513, 'rew_pos': -0.008403355545391513, 'rew_action': -0.0002681436538

In [6]:
    # 1. Создаем среду с базовыми параметрами визуализация
num_agents = 1 # Начнем с 2 квадрокоптеров
env = create_env(
        num_agents=num_agents,
        use_numba=False,
        episode_duration=15,
        quads_render=True
         )


Observation components: ['xyz', 'vxyz', 'R', 'omega']


/home/fire_tech/miniconda3/envs/swarm-rlc/lib/python3.11/site-packages/gymnasium/spaces/box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float32
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [8]:
# Запускаем тестовый эпизод с визуализацией

obs = env.reset()
env.render()  # Инициализируем окно рендеринга

Размеры изображения: 256x256
Размеры изображения: 256x256
Размеры изображения: 256x256
Размеры изображения: 256x256
Размеры изображения: 256x256
Размеры изображения: 256x256
Размеры изображения: 256x256
Размеры изображения: 256x256


In [9]:
# Задаем всем моторам 75% мощности (0.5) Взлет 
#constant_action = [0.2, 0.2, 0.2, 0.2]

for step in range(1000):
    # Генерируем случайные действия
    actions = [env.action_space.sample() for _ in range(1)]
    #actions = [constant_action,
               #constant_action
    #           ]  # Одинаковое действие для всех агентов
    obs, rewards, dones, infos = env.step(actions)
    
    # Отображаем текущее состояние
    env.render()
    
    # Выводим информацию каждые 20 шагов
    if step % 20 == 0:
        print(f"\nStep {step}:")
        print(f"Positions: {[o[:3] for o in obs]}")
        print(f"Velocities: {[o[3:6] for o in obs]}")
        print(f"Rewards: {rewards}")
    
    if any(dones):
        print("\nEpisode finished")
        break

env.close()


Step 0:
Positions: [array([ 1.50797143,  0.63638573, -0.68962101])]
Velocities: [array([-0.00452372, -0.01000486, -0.10240693])]
Rewards: [-0.0041688917547198865]

Step 20:
Positions: [array([ 1.49948024,  0.63773539, -0.78824983])]
Velocities: [array([-0.32600526, -0.26371781, -0.7975273 ])]
Rewards: [-0.012146259767211221]

Step 40:
Positions: [array([ 1.28162731,  0.50580334, -1.14135084])]
Velocities: [array([-1.62103098, -0.74598398, -2.94938272])]
Rewards: [-0.01612749409913865]

Step 60:
Positions: [array([ 0.89605528,  0.47473892, -1.94891338])]
Velocities: [array([0.04594521, 0.03096021, 0.0010817 ])]
Rewards: [-0.021091203853421545]

Step 80:
Positions: [array([ 0.95193628,  0.50624734, -1.9454229 ])]
Velocities: [array([0.35748644, 0.20195654, 0.01756819])]
Rewards: [-0.025061280085423753]

Step 100:
Positions: [array([ 0.93699431,  0.54441711, -1.95036059])]
Velocities: [array([-0.21219019,  0.39119664, -0.00119046])]
Rewards: [-0.02363731456271845]

Step 120:
Positions: [

In [12]:
from sample_factory.algo.utils.context import global_model_factory

from sample_factory.model.model_utils import fc_layer, nonlinearity
from gym_art.quadrotor_multi.quad_utils import QUADS_OBS_REPR
from types import SimpleNamespace


#from sample_factory.model.encoder import Encoder


from swarm_rl.models.quad_multi_model import (
    QuadMultiEncoder,
    QuadMultiHeadAttentionEncoder,
    QuadSingleHeadAttentionEncoder_Sim2Real
)


In [13]:

# Создаем среду
env = create_env(num_agents=1, use_numba=False)


Observation components: ['xyz', 'vxyz', 'R', 'omega']


In [14]:
cfg = SimpleNamespace(
    # Тип энкодера - базовый
    quads_encoder_type="basic",
    
    # Правильный формат состояния дрона
    quads_obs_repr='xyz_vxyz_R_omega',  # изменили здесь
    
    # Размер скрытого слоя
    rnn_size=256,
    
    # Функция активации
    nonlinearity='tanh',
    
    # Отключаем обработку препятствий
    quads_use_obstacles=False,
    
    # Параметры обработки соседей
    quads_neighbor_hidden_size=0,
    quads_neighbor_obs_type='none',
    quads_neighbor_visible_num=0,
    quads_num_agents=1,
    quads_neighbor_encoder_type='no_encoder',
    
    # Параметры препятствий
    quads_obstacle_obs_type='none',
    quads_obst_hidden_size=0
)



In [15]:
QUAD_BASELINE_CLI = (
    'python -m swarm_rl.train --env=quadrotor_multi --algo=APPO --use_rnn=False '
    '--serial_mode=True '  # добавляем этот параметр
    # остальные параметры оставляем как есть
)


In [16]:
from gym_art.quadrotor_multi.quad_utils import QUADS_OBS_REPR
print("Доступные форматы представления состояния:", QUADS_OBS_REPR.keys())


Доступные форматы представления состояния: dict_keys(['xyz_vxyz_R_omega', 'xyz_vxyz_R_omega_floor', 'xyz_vxyz_R_omega_wall'])


In [17]:
# Создаем среду
env = create_env(num_agents=1, use_numba=False)

# Получаем пространство наблюдений
obs_space = env.observation_space
print("Пространство наблюдений:", obs_space)



Observation components: ['xyz', 'vxyz', 'R', 'omega']
Пространство наблюдений: Box([ -8.  -8.  -5.  -3.  -3.  -3.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.
  -1. -40. -40. -40.], [ 8.  8.  5.  3.  3.  3.  1.  1.  1.  1.  1.  1.  1.  1.  1. 40. 40. 40.], (18,), float32)


In [18]:
from swarm_rl.models.quad_multi_model import QuadMultiEncoder

model = QuadMultiEncoder(cfg, obs_space)

In [19]:
# 1. Общая структура модели
print("Архитектура модели:")
print(model)

# 2. Количество параметров
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nВсего параметров: {total_params}")
print(f"Обучаемых параметров: {trainable_params}")

# 3. Размеры входов/выходов
print(f"\nРазмерность входного состояния: {model.self_obs_dim}")
print(f"Размерность выходного вектора: {model.encoder_out_size}")

# 4. Структура энкодера собственного состояния
print("\nСтруктура self_encoder:")
for idx, layer in enumerate(model.self_encoder):
    print(f"Слой {idx}: {layer}")

# 5. Структура финального преобразования
print("\nСтруктура feed_forward:")
for idx, layer in enumerate(model.feed_forward):
    print(f"Слой {idx}: {layer}")


Архитектура модели:
QuadMultiEncoder(
  (self_encoder): Sequential(
    (0): Linear(in_features=18, out_features=256, bias=True)
    (1): Tanh()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): Tanh()
  )
  (feed_forward): Sequential(
    (0): Linear(in_features=256, out_features=512, bias=True)
    (1): Tanh()
  )
)

Всего параметров: 202240
Обучаемых параметров: 202240

Размерность входного состояния: 18
Размерность выходного вектора: 512

Структура self_encoder:
Слой 0: Linear(in_features=18, out_features=256, bias=True)
Слой 1: Tanh()
Слой 2: Linear(in_features=256, out_features=256, bias=True)
Слой 3: Tanh()

Структура feed_forward:
Слой 0: Linear(in_features=256, out_features=512, bias=True)
Слой 1: Tanh()


In [20]:

# Получаем состояние из среды
state = env.reset()
# Создаем правильный формат входных данных - словарь с ключом 'obs'
obs_dict = {'obs': torch.FloatTensor(state)}
print(obs_dict)

{'obs': tensor([[-7.7764e-01,  4.1949e-01, -7.2464e-01, -1.2967e-02, -7.0288e-03,
          7.3514e-03,  1.0682e-01,  9.9428e-01,  0.0000e+00, -9.9428e-01,
          1.0682e-01, -0.0000e+00, -0.0000e+00,  0.0000e+00,  1.0000e+00,
          2.4092e-05,  2.5328e-04,  4.1404e-05]])}


/tmp/ipykernel_880/3291676292.py:4: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  obs_dict = {'obs': torch.FloatTensor(state)}


In [21]:

 # Пропускаем через энкодер
encoded_state = model(obs_dict)

print("Размер закодированного состояния:", encoded_state.shape)



Размер закодированного состояния: torch.Size([1, 512])


---------------------------------------------

In [12]:
# Создаем среду
env = create_env(num_agents=1, use_numba=False)

# Получаем пространство наблюдений
obs_space = env.observation_space
print("Пространство наблюдений:", obs_space)

Observation components: ['xyz', 'vxyz', 'R', 'omega']
Пространство наблюдений: Box([ -8.  -8.  -5.  -3.  -3.  -3.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.
  -1. -40. -40. -40.], [ 8.  8.  5.  3.  3.  3.  1.  1.  1.  1.  1.  1.  1.  1.  1. 40. 40. 40.], (18,), float32)


/home/fire_tech/miniconda3/envs/swarm-rlc/lib/python3.11/site-packages/gymnasium/spaces/box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float32
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [13]:
obs = env.reset()
print("Наблюдения:", obs)
print("Тип наблюдений:", type(obs))
print("Длина списка наблюдений:", len(obs))

Наблюдения: [array([-1.79409262e+00,  1.51630572e+00,  1.18771311e+00,  1.05227357e-02,
        1.18552215e-03,  8.80967455e-03,  3.97366697e-01,  9.17659909e-01,
        0.00000000e+00, -9.17659909e-01,  3.97366697e-01, -0.00000000e+00,
       -0.00000000e+00,  0.00000000e+00,  1.00000000e+00,  1.81327431e-04,
       -7.14896134e-05, -1.43510493e-04])]
Тип наблюдений: <class 'list'>
Длина списка наблюдений: 1


In [14]:
# Создаем случайное действие для тестирования
action = [np.random.uniform(-1, 1, size=4)]  # 4 мотора квадрокоптера

# Делаем шаг в среде
next_obs, reward, done, info = env.step(action)

print("Награда:", reward)
print("Эпизод завершен:", done)
print("Следующее наблюдение:", next_obs)


Награда: [-0.008533733194740415]
Эпизод завершен: [False]
Следующее наблюдение: [array([-1.79998798e+00,  1.52178092e+00,  1.18754542e+00,  4.95319358e-03,
       -2.25934212e-02, -9.27058877e-02,  3.97388345e-01,  9.17650534e-01,
        3.39256462e-05, -9.17650534e-01,  3.97388346e-01, -3.16090881e-05,
       -4.24877530e-05, -1.85708041e-05,  9.99999999e-01, -1.05473217e-02,
        3.75349188e-03,  1.70600607e-02])]


In [23]:
# Основной цикл симуляции
max_steps = 5000
total_reward = 0

for step in range(max_steps):
    action = [np.random.uniform(-1, 1, size=4)]
    obs, reward, done, info = env.step(action)
    
    total_reward += reward[0]
    
    if done[0]:
        print(f"Эпизод завершен на шаге {step}")
        print(f"Суммарная награда: {total_reward}")
        break

    if step % 100 == 0:
        print(f"Шаг {step}, текущая награда: {reward[0]}")


Шаг 0, текущая награда: -0.00017141411652697752
Шаг 100, текущая награда: -0.01178555692920897
Шаг 200, текущая награда: -0.02569477300951618
Шаг 300, текущая награда: -0.022212671823887413
Шаг 400, текущая награда: -0.027958545335819675
Шаг 500, текущая награда: -0.020842785402973404
Шаг 600, текущая награда: -0.02097506136640617
Шаг 700, текущая награда: -0.021301705674226082
Шаг 800, текущая награда: -0.025979353778298464
Шаг 900, текущая награда: -0.028209967757529226
Шаг 1000, текущая награда: -0.030530863400835093
Шаг 1100, текущая награда: -0.03531070896856892
Шаг 1200, текущая награда: -0.02230657041938155
Шаг 1300, текущая награда: -0.025739332721441823
Шаг 1400, текущая награда: -0.028483972749515665
Эпизод завершен на шаге 1500
Суммарная награда: -34.510498469660675
